# 01 — Where do the 4-bit Parakeet encoder's bytes and milliseconds go?

Track A of [#271](https://github.com/rahulsharmavishwakarma/vesper/issues/271). Research only: nothing here
changes the app.

**Question.** The shipped Parakeet TDT 0.6B v3 4-bit weight-only encoder is 679 MB against 652 MB for the old INT8 and decodes
about 1.5x slower. A 4-bit 0.6B model should be far smaller. **Hypothesis (unverified):** ONNX Runtime's `MatMulNBits`
only covers `MatMul`, so pointwise (1x1) and depthwise convolutions stay FP32, and that is where the size and time sit.

**This notebook measures:**
1. Bytes per operator class in the FP32 and 4-bit encoders, with convolutions split into pointwise, depthwise and other.
2. Median encoder time for FP32 and 4-bit across thread counts (CPU).
3. Per-operator time from the ONNX Runtime profiler.
4. Whether `accuracy_level=4` (int8 compute inside `MatMulNBits`) changes size or time.

**Notes.** CPU only; a GPU runtime is not needed. Colab CPU timings are **not comparable** with desktop numbers in the app's
reports; compare variants only against each other inside one run. Inputs are random features of 10 s, so this measures speed,
not accuracy. Only public model files are used. Needs about 8 GB of RAM and 8 GB of disk; takes about 15 minutes.

In [ ]:
%%bash
set -e
# Exported with `uv export --locked` from benchmarks/asr/python/onnx-quantize (default group); do not edit by hand.
# Colab already ships uv; install it only if it is missing.
command -v uv >/dev/null || python -m pip install -q uv
cat > /tmp/requirements.txt <<'REQUIREMENTS'
annotated-doc==0.0.5 \
    --hash=sha256:117bac03a25ede5df5440e855b32d556049ca169ead221505badf432fed4b101 \
    --hash=sha256:c7e58ce09192557605d8bbd92836d7e1d520ac9580096042c0bfd197efacf1bb
    # via typer
anyio==4.15.1 \
    --hash=sha256:6152fdbbf9a77fdec97731721bebf7c4c44f7c29b424b0065826173efc7ed101 \
    --hash=sha256:9f28306018cbd6d329e64a36d58256edff76dd996fe423bc957326e578b82a94
    # via httpx
certifi==2026.10.19 \
    --hash=sha256:035c1e85572b141bf1c3a3eff0e33a8eb74917522f2559075ff8d2fea780a1ae \
    --hash=sha256:42d35340cc5b9ba78e818ddbff0c5b1c2b779de448a340fd1b989f5fa8670e1b
    # via
    #   httpcore
    #   httpx
colorama==0.4.6 ; sys_platform == 'win32' \
    --hash=sha256:08695f5cb7ed6e0531a20572697297273c47b8cae5a63ffc6d6ed5c201be6e44 \
    --hash=sha256:4f1d9991f5acc0ca119f9d443620b77f9d6b33703e51011c16baf57afb285fc6
    # via
    #   tqdm
    #   typer
filelock==4.2.4 \
    --hash=sha256:301241438c57936ef93f4243a94f2559022268bfcf9b375a9a72eb0f8db8358e \
    --hash=sha256:c6d8663535e1650ee430507b4ffbac7b464dd64b7647ea0112a9f406499d163d
    # via huggingface-hub
flatbuffers==25.12.19 \
    --hash=sha256:7634f50c427838bb021c2d66a3d1168e9d199b0607e6329399f04846d42e20b4
    # via onnxruntime
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via huggingface-hub
h11==0.16.0 \
    --hash=sha256:4e35b956cf45792e4caa5885e69fba00bdbc6ffafbfa020300e549b208ee5ff1 \
    --hash=sha256:63cf8bbe7522de3bf65932fda1d9c2772064ffb3dae62d55932da54b31cb6c86
    # via httpcore
hf-xet==1.7.0 ; platform_machine == 'AMD64' or platform_machine == 'aarch64' or platform_machine == 'amd64' or platform_machine == 'arm64' or platform_machine == 'x86_64' \
    --hash=sha256:0a9e802f33bf50c851abe45fc5380e61f959e2d369647d6742b79ad9d6c27cab \
    --hash=sha256:2814a6e999d13464c4d679b788cc5d784eb5a4edfc638a31f10e9a11ab531ef8 \
    --hash=sha256:2b7bb5727889b0f2436dbaaad8fc4c3e66b8240d992716989e0c086b4278b1bc \
    --hash=sha256:59fba37039233c7fcbe196817d6cdcf1b40dfb17b410f229d85b0cf0a1848da4 \
    --hash=sha256:80f79dae613ce9e0ea1fd1ae15616ca9ac74aed4c770aabc199c4f03ebecc863 \
    --hash=sha256:acc3851cf2576a8fb2ae926da863f4efabe21303cf292e9a44332802ab0dcc6a \
    --hash=sha256:d406ec79053c0871817f700c2ac8c36ba0d87f9c34b7458b0f0063bb218b0466 \
    --hash=sha256:e3e88a7a75d7d95cbee1f37dc31341d6201124cf21c6c4b1dfab8ccba9b09e0f \
    --hash=sha256:fcfd6c22418e57dd5b3aea649e813b2e2cfb2aebf317b210d90f1fe4b3018b52
    # via huggingface-hub
httpcore==1.0.9 \
    --hash=sha256:2d400746a40668fc9dec9810239072b40b4484b640a8c38fd654a024c7a1bf55 \
    --hash=sha256:6e34463af53fd2ab5d807f399a9b45ea31c3dfa2276f15a2c3f00afff6e176e8
    # via httpx
httpx==0.28.1 \
    --hash=sha256:75e98c5f16b0f35b567856f597f06ff2270a374470a5c2392242528e3e3e42fc \
    --hash=sha256:d909fcccc110f8c7faf814ca82a9a4d816bc5a6dbfea25d6591d6985b8ba59ad
    # via huggingface-hub
huggingface-hub==1.5.0 \
    --hash=sha256:c9c0b3ab95a777fc91666111f3b3ede71c0cdced3614c553a64e98920585c4ee \
    --hash=sha256:f281838db29265880fb543de7a23b0f81d3504675de82044307ea3c6c62f799d
    # via vesper-onnx-quantize
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
    # via
    #   anyio
    #   httpx
markdown-it-py==4.2.0 \
    --hash=sha256:04a21681d6fbb623de53f6f364d352309d4094dd4194040a10fd51833e418d49 \
    --hash=sha256:9f7ebbcd14fe59494226453aed97c1070d83f8d24b6fc3a3bcf9a38092641c4a
    # via rich
mdurl==0.1.2 \
    --hash=sha256:84008a41e51615a49fc9966191ff91509e3c40b939176e643fd50a5c2196b8f8 \
    --hash=sha256:bb413d29f5eea38f31dd4754dd7377d4465116fb207585f97bf925588687c1ba
    # via markdown-it-py
ml-dtypes==0.6.0 \
    --hash=sha256:084dfe51a7ad58b171f05115f8226ed4233a454a1611371947e806e76f0c638d \
    --hash=sha256:26b1f1fa4f0435a2946859823f6e2bf06796f1e9f10f5a05b08a5e3c8f46ff69 \
    --hash=sha256:28d676428b104bb9717b0928bc5c5129f2d6b51b6727587cc4289e7bf8713cb5 \
    --hash=sha256:2a3e9d53925597fbffafd2a37048dadeddd0bdaba58058f6ae0869ed709a184d \
    --hash=sha256:37da32aa97749251025666d62372775019594577b9c9e9cfda83bed48d778fdb \
    --hash=sha256:3b4a480aa8fd54a1805b8ac10f3f91763926a74f73c0c364c10f9231854f4170 \
    --hash=sha256:5359c588cc62de6f78d7430f06b65853d884955494d86d6ad90b6dd64a3f3a08 \
    --hash=sha256:57ed0d6b4ac5e7868361303a9c57fbcf63b768236ee14456f585dfcf260d0292 \
    --hash=sha256:5e60251d32ced5598972e4d5e06a2f044341f9291402551a3f6f0ec44f9299b0 \
    --hash=sha256:6eaed129a4afe90694b8685e2f9b6294849f5eda4af9a15be83a4326eeebd775 \
    --hash=sha256:fb87f46b4f7ad7b5d3ad8f4b452b024bd4229d44c8ff934798c1fe656210387a
    # via
    #   onnx
    #   onnx-ir
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via sympy
numpy==2.5.4 \
    --hash=sha256:1c2e71b04c6cad90026e544501bbe0ab9290fa8a4d845e7e8c0d124fb429c988 \
    --hash=sha256:2377da2dd3ba2c1200956acbab2a358c83b8e1f8531191672d1cd6ad83250d53 \
    --hash=sha256:2fa3328f784fc8277fc48026f6cad516f5c561c5d8e2e39b3c9e0c8f23223b34 \
    --hash=sha256:381a7a3d2e65e64c0ec302795ab9dc12bb1e73f150904699c153716177eebdaf \
    --hash=sha256:5258bc06526964be5face2fc6f756857a3f24f21ec3e72ca131337a75b165d6c \
    --hash=sha256:6d6a71b9d9a97c03633aa12565ef2825ffa036cc1d99cfd50dacf0f128af4fe2 \
    --hash=sha256:6ffa07666f8da0eef81d149934a626d0d95fbd6838432a33e66245423a9062c0 \
    --hash=sha256:7415db95818b39ec475a5eea54d9e3b6bc83e3912158e46da3438cdce399804d \
    --hash=sha256:8b4d2fd2d34e5f8c9235ee787de5631a37a28402b15cb80814df973d2be54129 \
    --hash=sha256:9968ab7e49b93ac6e1c3b2239732183152c9150f16308d30b66a372cffe3483c \
    --hash=sha256:9a94cf751c9ad8ebaa835bcd3d40dacf8534ad086b88c38029b65123c7999d2a \
    --hash=sha256:9cb18a327b49c5c337f972b03682f6a49855525faaf3c0d3e9c96cd0fd8880a8 \
    --hash=sha256:a7b1b6353e36a7e50de2973a38d705c88ee93adcf120673cee7f45a4a3fa223a \
    --hash=sha256:aa1cce2ff3f8d953de38b76bf44602caeb69f101430208f64a10067f7cb4b1d3 \
    --hash=sha256:aec3fc4b32ff82421274f5d205c559c51c840c8df66a78efd7f3612dd005a26a \
    --hash=sha256:b11e8fda06a7d69f15ebf542660b74466c2e51094800c1fb794f47ad4faeef17 \
    --hash=sha256:b86966fbe4ad7de710422175572bcdc75fdedadfb54bc6fab7deabccddd7780b \
    --hash=sha256:b89d0aaae2fe498c648f4c4795c084db535af5bd98ef942b2a3681fb74ce8645 \
    --hash=sha256:bc39ac66a7a9a3fbd6134fda43136b60ffde99c8f4501e64e0d2b24da137babf \
    --hash=sha256:c6342f54c67093cae5c0227eb0eb772fdb79f2a2c37a6eb278b9909ee06aa356 \
    --hash=sha256:d8200f16437b289a5bb927c6e184eccc3e8389bc0070fea4cd5b9e13c1757959 \
    --hash=sha256:fbde6962867ee75b48b0ee29b2b9372ec5d617799dbaf38e82dc0596f2f7738a \
    --hash=sha256:fe4d21ab149f15e4e6043dfb0de87e6e5f34ac176cde83060e9802981fca2ac2
    # via
    #   ml-dtypes
    #   onnx
    #   onnx-ir
    #   onnxruntime
onnx==1.23.2 \
    --hash=sha256:008cb0467b2bbee41448acc7da8b6f4e704624cb0d327a2d5adafc7ce19bc5b8 \
    --hash=sha256:1b8680ce1e6a9a4736374a9dce4de14ea8ee05e0dccf0784a78a6e5646bdc1f6 \
    --hash=sha256:7abf381d278f31ac62487fddedc9dd42da842dce94d5d43536836ee3efdf4a2b \
    --hash=sha256:80cef0fad59524d02c21ec93f4fbccdcc6223f1c33339d597519a2d27cac19a7 \
    --hash=sha256:9b382ba898a7c142a0801d03cf04ecabced96c1543c7b643a86f0928143802de \
    --hash=sha256:a203efdbaabbbe8f25e854e2b2921382d6fcf4c67895656f939044b0632974e8 \
    --hash=sha256:b0b8dae0d33dd8606370bc264b0b1d6e64cfdf8b83d7c676fab8eff6b88ca409 \
    --hash=sha256:e79e35e152d3095c6910ae81013bbc68679e32bfc0ca76f840968d4b6fdfb864
    # via
    #   onnx-ir
    #   vesper-onnx-quantize
onnx-ir==1.1.0 \
    --hash=sha256:76cc313a788c073b75271c1b6abb172aaf90fcc6335ed40f9f3238b2064a2fa3 \
    --hash=sha256:b72176de575d820ec26864d17270d58ef886a7cf64db7649d15fcc3d5ae2a8ee
    # via vesper-onnx-quantize
onnxruntime==1.31.0 \
    --hash=sha256:09d56445c1753e66e0912de69d3f0184016ad9a191dcd6925bf5dd570d2bfbe5 \
    --hash=sha256:0ba02a44acb6203040354d9a1f160e3f37a43feac7bb05caa3e0ea545efed505 \
    --hash=sha256:35758d7606d578ec5b9d65f6e8a1f488013194c3f6097038a3223cb26d35ef9a \
    --hash=sha256:37fd78cee5160c7a43a1730ccb3682ffd880af9c9e80385d625c0c2f8b125809 \
    --hash=sha256:5c54a0eb7b2b4eef3eb9dcfaf82f5ce880db07288dc309574f6657e9da5cc754 \
    --hash=sha256:5e129d6c56abd53e659cb70f00a108d6824086470ff99c2e47a82e5786563db3 \
    --hash=sha256:73e0165d58ece068c2a8a1c477c90b38e5a8adbbd399fdfdfd4bd79cbc28ff8d \
    --hash=sha256:aaab9b3af536b06ca27ab5e35e3d429c97457ce76cf298af103f687e8b9975c0 \
    --hash=sha256:ad663106f6eeff3d454f24a786450459d07f30e74863851104fc1b8b3f368127 \
    --hash=sha256:e0e050bf9ec754950a6ba9830e4032f4004d972c6f38c5642fef26d44d894965 \
    --hash=sha256:e51d10d2e2e1e5bbf9b126a0cd9853d3e6c4e21424518dd50160b91471be33dc \
    --hash=sha256:e93d7c5fad20afa697ac16f376fd0306ed180f9a376e86106cc0b7d84f53ef87
    # via vesper-onnx-quantize
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
    # via
    #   huggingface-hub
    #   onnx-ir
    #   onnxruntime
protobuf==7.36.2 \
    --hash=sha256:497d0463ff3316681da6c0b9e8d06cb465d61abce00b613ab42226175644d1bb \
    --hash=sha256:89f23aa53c24553a2416fd4fd1ec06f74fa42b14b546d8883128813f775bbfd2 \
    --hash=sha256:912c1221170e16c08d1f086762f563dd61ff83c18b5fa6652952dfaded66f728 \
    --hash=sha256:a300819d441e078a5608c0d3c709796bb548136058fda017ae51d425b44fd353 \
    --hash=sha256:bdb3a345d48db958e6ce1f18e508beb0cc981d64f24088427549c866cd039f1e \
    --hash=sha256:cbc70b17ee27e28894c7fee8bb04be1abead49e936bc70eb60052531eee2079e \
    --hash=sha256:e11e1f0180583a2af89db6a2ecd9e8dc40aa6d2988ca175bfd0e6d12ea72d74e \
    --hash=sha256:f4fee11ec330d238b34a05c9b675f693c20415d1c5bd7d5320cc2f8a798eb9cf
    # via
    #   onnx
    #   onnxruntime
pygments==2.21.0 \
    --hash=sha256:2363c69b61c4a97c838da3b130dcd6468f4848992b21a82f2a63ec34377137d9 \
    --hash=sha256:610ca751c9bc2492b38eb9a38a7fbc93edbbb2d7182edaf34e66ae493dee5c8c
    # via rich
pyyaml==6.0.3 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
    # via huggingface-hub
rich==15.0.0 \
    --hash=sha256:33bd4ef74232fb73fe9279a257718407f169c09b78a87ad3d296f548e27de0bb \
    --hash=sha256:edd07a4824c6b40189fb7ac9bc4c52536e9780fbbfbddf6f1e2502c31b068c36
    # via typer
shellingham==1.5.4 \
    --hash=sha256:7ecfff8f2fd72616f7481040475a65b2bf8af90a56c89140852d1120324e8686 \
    --hash=sha256:8dbca0739d487e5bd35ab3ca4b36e11c4078f3a234bfce294b0a0291363404de
    # via typer
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via onnx-ir
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
    # via huggingface-hub
typer==0.27.3 \
    --hash=sha256:d0396f770a560ab1b0a8504e13b5f254b728cedb05c61cf0359e944e50ce8901 \
    --hash=sha256:e50022f28b82a86313e54501317a1db64bf8f8d036ff8cfe5ca7e47675454aff
    # via huggingface-hub
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via
    #   anyio
    #   huggingface-hub
    #   onnx
    #   onnx-ir
REQUIREMENTS
uv pip install --system -q --require-hashes -r /tmp/requirements.txt

In [ ]:
import gc
import hashlib
import json
import os
import platform
import statistics
import sys
import time
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
from huggingface_hub import hf_hub_download

WORK = Path(os.environ.get("NB_WORK", "/content/work"))
RESULTS = Path(os.environ.get("NB_RESULTS", "/content/results"))
WORK.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

# Pinned sources. FP32 is the official sherpa-onnx export; WO4 is the app's current build (#251).
FP32_REPO, FP32_REV = "csukuangfj/sherpa-onnx-nemo-parakeet-tdt-0.6b-v3", "1a468a35cbba69418f126de829e75261dea4a4e4"
WO4_REPO, WO4_REV = (
    "rahulsharmavishwakarma/vesper-parakeet-tdt-0.6b-v3-sherpa-onnx-wo4",
    "8e178c8a971286a2d849196e47da4e776c82d43b",
)


def fetch(repo, rev, name):
    """Download one file as a plain file (not a cache symlink), so ONNX external data resolves."""
    return Path(hf_hub_download(repo, name, revision=rev, local_dir=WORK / "src" / repo / rev))


def environment():
    cpu, ram_gb = "", None
    try:
        cpu = next(l for l in open("/proc/cpuinfo") if l.startswith("model name")).split(":", 1)[1].strip()
        ram_gb = round(int(next(l for l in open("/proc/meminfo") if l.startswith("MemTotal")).split()[1]) / 1048576, 1)
    except Exception:
        cpu = cpu or platform.processor()
    return {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "cpu": cpu,
        "logical_cpus": os.cpu_count(),
        "ram_gb": ram_gb,
        "onnxruntime": ort.__version__,
        "onnx": onnx.__version__,
        "providers": ort.get_available_providers(),
        "sources": {"fp32": [FP32_REPO, FP32_REV], "wo4": [WO4_REPO, WO4_REV]},
    }


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 24), b""):
            digest.update(chunk)
    return digest.hexdigest()


def save_results(name, payload):
    payload = {"environment": environment(), **payload}
    path = RESULTS / name
    path.write_text(json.dumps(payload, indent=2) + "\n")
    print("saved", path)
    return path


print(json.dumps(environment(), indent=2))

In [ ]:
fp32_encoder = fetch(FP32_REPO, FP32_REV, "encoder.onnx")
fetch(FP32_REPO, FP32_REV, "encoder.weights")  # external tensor data, must sit next to encoder.onnx
wo4_encoder = fetch(WO4_REPO, WO4_REV, "encoder.onnx")
for path in (fp32_encoder, wo4_encoder):
    print(path.name, path.parent.name[:8], f"{path.stat().st_size:,} bytes (graph file only)")

## 1. Bytes by operator class

In [ ]:
from collections import defaultdict
from onnx import TensorProto, helper


def tensor_bytes(tensor):
    """Size of an initializer without loading external data."""
    if tensor.data_location == TensorProto.EXTERNAL:
        info = {entry.key: entry.value for entry in tensor.external_data}
        if "length" in info:
            return int(info["length"])
    if tensor.raw_data:
        return len(tensor.raw_data)
    count = int(np.prod(tensor.dims)) if tensor.dims else 1
    width = {TensorProto.FLOAT: 4, TensorProto.INT64: 8, TensorProto.INT32: 4, TensorProto.UINT8: 1,
             TensorProto.INT8: 1, TensorProto.FLOAT16: 2, TensorProto.DOUBLE: 8}.get(tensor.data_type, 4)
    return count * width


def conv_class(node, shapes):
    weight = shapes.get(node.input[1])
    if weight is None:
        return "Conv:unknown"
    group = next((helper.get_attribute_value(a) for a in node.attribute if a.name == "group"), 1)
    kernel = list(weight[2:])
    if all(k == 1 for k in kernel) and group == 1:
        return "Conv:pointwise(1x1)"
    if group == weight[0] and group > 1:
        return "Conv:depthwise"
    return "Conv:other"


def breakdown(path):
    model = onnx.load(str(path), load_external_data=False)
    graph = model.graph
    inits = {t.name: t for t in graph.initializer}
    shapes = {name: list(t.dims) for name, t in inits.items()}
    owner = {}
    counts = defaultdict(int)
    for node in graph.node:
        label = conv_class(node, shapes) if node.op_type == "Conv" else node.op_type
        counts[label] += 1
        for name in node.input:
            if name in inits and name not in owner:
                owner[name] = label
    rows = defaultdict(lambda: {"bytes": 0, "tensors": 0})
    for name, tensor in inits.items():
        label = owner.get(name, "(unused or shared)")
        rows[label]["bytes"] += tensor_bytes(tensor)
        rows[label]["tensors"] += 1
    total = sum(r["bytes"] for r in rows.values())
    table = [
        {"class": k, "nodes": counts.get(k, 0), "tensors": v["tensors"], "bytes": v["bytes"],
         "share": round(v["bytes"] / total, 4) if total else 0}
        for k, v in sorted(rows.items(), key=lambda kv: -kv[1]["bytes"])
    ]
    return {"total_initializer_bytes": total, "classes": table}


def show(label, result):
    print(f"\n{label}: {result['total_initializer_bytes']:,} initializer bytes")
    print(f"{'class':28} {'nodes':>6} {'tensors':>8} {'MB':>9} {'share':>7}")
    for row in result["classes"][:14]:
        print(f"{row['class']:28} {row['nodes']:>6} {row['tensors']:>8} {row['bytes'] / 1e6:>9.1f} {row['share'] * 100:>6.1f}%")


size_fp32 = breakdown(fp32_encoder)
size_wo4 = breakdown(wo4_encoder)
show("FP32 encoder", size_fp32)
show("4-bit weight-only encoder (current app build)", size_wo4)

**How to read it.** In the 4-bit table, the `MatMulNBits` rows are the compressed weights. Large remaining rows under
`Conv:pointwise(1x1)`, `Conv:depthwise`, `Conv:other` or `MatMul` mean those weights are still FP32 and uncovered.
If `Conv:pointwise(1x1)` dominates, the rewrite in notebook 02 is worth testing.

## 2. Encoder time versus thread count

In [ ]:
def make_feed(session, frames=1000, seed=0):
    """Random 10 s of log-mel-like features. Only for timing; accuracy is measured in notebook 03."""
    rng = np.random.default_rng(seed)
    feed = {}
    for item in session.get_inputs():
        shape = [d if isinstance(d, int) else (1 if i == 0 else frames) for i, d in enumerate(item.shape)]
        if "int" in item.type:
            feed[item.name] = np.full(shape, frames, dtype=np.int64)
        else:
            feed[item.name] = rng.standard_normal(shape).astype(np.float32)
    return feed


def open_session(path, threads, profile=False):
    options = ort.SessionOptions()
    options.intra_op_num_threads = threads
    options.inter_op_num_threads = 1
    options.enable_profiling = profile
    return ort.InferenceSession(str(path), options, providers=["CPUExecutionProvider"])


def time_encoder(path, threads, runs=5, warmup=2):
    session = open_session(path, threads)
    feed = make_feed(session)
    for _ in range(warmup):
        session.run(None, feed)
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        session.run(None, feed)
        times.append((time.perf_counter() - start) * 1000)
    del session
    return {"median_ms": round(statistics.median(times), 1), "min_ms": round(min(times), 1), "runs": runs}

In [ ]:
cpus = os.cpu_count() or 2
THREADS = sorted({1, 2, 4, cpus})
timing = {"fp32": {}, "wo4": {}}
for label, path in (("fp32", fp32_encoder), ("wo4", wo4_encoder)):
    for threads in THREADS:
        timing[label][threads] = time_encoder(path, threads)
        print(label, threads, "threads:", timing[label][threads])

## 3. Per-operator time (ONNX Runtime profiler)

Runs 5 inferences with profiling on and sums event durations by operator. Fused nodes appear under their fused name.

In [ ]:
from collections import Counter


def profile_ops(path, threads, runs=5):
    session = open_session(path, threads, profile=True)
    feed = make_feed(session)
    for _ in range(runs + 1):
        session.run(None, feed)
    profile_path = session.end_profiling()
    events = json.loads(Path(profile_path).read_text())
    totals = Counter()
    for event in events:
        if event.get("cat") == "Node" and event.get("name", "").endswith("_kernel_time"):
            totals[event.get("args", {}).get("op_name", "?")] += event["dur"]
    Path(profile_path).unlink()
    del session
    total = sum(totals.values()) or 1
    return [{"op": op, "ms_per_run": round(us / 1000 / (runs + 1), 1), "share": round(us / total, 4)}
            for op, us in totals.most_common(12)]


PROFILE_THREADS = min(4, cpus)
profiles = {}
for label, path in (("fp32", fp32_encoder), ("wo4", wo4_encoder)):
    profiles[label] = profile_ops(path, PROFILE_THREADS)
    print(f"\n{label} at {PROFILE_THREADS} threads")
    for row in profiles[label]:
        print(f"  {row['op']:24} {row['ms_per_run']:>9.1f} ms/run  {row['share'] * 100:>5.1f}%")

## 4. Does `accuracy_level` change speed or size?

`accuracy_level` is an attribute on each `MatMulNBits` node (`None`/unset is the app's current build; `4` asks the kernel
to compute in int8; `1` computes in FP32). It does not change the stored weights, so this cell edits the attribute on the
current build and times the result, instead of re-quantizing. A matching local check on a re-quantized model gave the same
file size within about 5 KB.

In [ ]:
ACCURACY_LEVELS = [None, 1, 4]
BEST_THREADS = min(os.cpu_count() or 2, 8)


def with_accuracy_level(source, destination, level):
    model = onnx.load(str(source))
    for node in model.graph.node:
        if node.op_type != "MatMulNBits":
            continue
        kept = [a for a in node.attribute if a.name != "accuracy_level"]
        del node.attribute[:]
        node.attribute.extend(kept)
        if level is not None:
            node.attribute.extend([helper.make_attribute("accuracy_level", level)])
    destination.parent.mkdir(parents=True, exist_ok=True)
    onnx.save_model(model, str(destination))
    del model
    gc.collect()


accuracy_runs = {}
for level in ACCURACY_LEVELS:
    out = WORK / "variants-01" / f"accuracy-{level}" / "encoder.onnx"
    if not out.exists():
        with_accuracy_level(wo4_encoder, out, level)
    accuracy_runs[str(level)] = {
        "bytes": out.stat().st_size,
        "timing_by_threads": {t: time_encoder(out, t) for t in sorted({1, BEST_THREADS})},
        "sha256": sha256(out),
    }
    print("accuracy_level", level, accuracy_runs[str(level)])
print("current app encoder bytes:", wo4_encoder.stat().st_size, "sha256", sha256(wo4_encoder)[:16])

## Save and download the results

In [ ]:
path = save_results(
    "01-wo4-graph-diagnosis.json",
    {
        "notebook": "01-wo4-graph-diagnosis",
        "issue": "https://github.com/rahulsharmavishwakarma/vesper/issues/271",
        "bytes_by_class": {"fp32": size_fp32, "wo4": size_wo4},
        "timing_by_threads": timing,
        "profile_threads": PROFILE_THREADS,
        "profile_top_ops": profiles,
        "accuracy_level_runs": accuracy_runs,
        "input": "random features, 1000 frames; timing only",
    },
)
try:
    from google.colab import files

    files.download(str(path))
except ImportError:
    pass